In [0]:
customers_path = "/Volumes/workspace/bronze/ecommerce_data/customers.csv"
orders_path = "/Volumes/workspace/bronze/ecommerce_data/orders.csv"

customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(customers_path)
)

orders_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(orders_path)
)

display(customers_df)
display(orders_df)

customer_id,customer_name,email,city,state,signup_date,customer_segment,is_active
1,Ravi Reddy,customer001@example.com,Pune,Maharashtra,2025-01-06,VIP,true
2,Sita Reddy,customer002@example.com,Vizag,Andhra Pradesh,2025-01-11,Premium,true
3,Kiran Reddy,customer003@example.com,Hyderabad,Telangana,2025-01-16,Regular,true
4,Priya Reddy,customer004@example.com,Mumbai,Maharashtra,2025-01-21,New,true
5,Arun Reddy,customer005@example.com,Kolkata,West Bengal,2025-01-26,VIP,true
6,Ananya Reddy,customer006@example.com,Jaipur,Rajasthan,2025-01-31,Premium,true
7,Rahul Reddy,customer007@example.com,Chennai,Tamil Nadu,2025-02-05,Regular,false
8,Sneha Reddy,customer008@example.com,Delhi,Delhi,2025-02-10,New,true
9,Vikram Reddy,customer009@example.com,Coimbatore,Tamil Nadu,2025-02-15,VIP,true
10,Meera Reddy,customer010@example.com,Bangalore,Karnataka,2025-02-20,Premium,true


order_id,customer_id,product_name,product_category,quantity,unit_price,order_date,order_status
1001,4,Laptop,Computers,1,950,2026-01-01,Delivered
1002,11,Phone,Mobiles,2,1087,2026-01-04,Cancelled
1003,18,Headphones,Audio,3,1224,2026-01-07,Pending
1004,25,Monitor,Computers,4,1361,2026-01-10,Shipped
1005,32,Keyboard,Accessories,5,1498,2026-01-13,Delivered
1006,39,Mouse,Accessories,1,1635,2026-01-16,Cancelled
1007,46,Tablet,Mobiles,2,1772,2026-01-19,Pending
1008,53,Smartwatch,Wearables,3,1909,2026-01-22,Shipped
1009,60,Speaker,Audio,4,2046,2026-01-25,Delivered
1010,67,Camera,Cameras,5,2183,2026-01-28,Cancelled


In [0]:
print("Customer rows:", customers_df.count())
print("Customer columns:", len(customers_df.columns))

print("Order rows:", orders_df.count())
print("Order columns:", len(orders_df.columns))


Customer rows: 100
Customer columns: 8
Order rows: 100
Order columns: 8


In [0]:
customers_df.printSchema()
orders_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- is_active: boolean (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)



In [0]:
from pyspark.sql import functions as F

customers_df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in customers_df.columns
]).show()

orders_df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in orders_df.columns
]).show()

+-----------+-------------+-----+----+-----+-----------+----------------+---------+
|customer_id|customer_name|email|city|state|signup_date|customer_segment|is_active|
+-----------+-------------+-----+----+-----+-----------+----------------+---------+
|          0|            0|    0|   0|    0|          0|               0|        0|
+-----------+-------------+-----+----+-----+-----------+----------------+---------+

+--------+-----------+------------+----------------+--------+----------+----------+------------+
|order_id|customer_id|product_name|product_category|quantity|unit_price|order_date|order_status|
+--------+-----------+------------+----------------+--------+----------+----------+------------+
|       0|          0|           0|               0|       0|         0|         0|           0|
+--------+-----------+------------+----------------+--------+----------+----------+------------+



In [0]:
# Check completely duplicate rows
print(
    "Duplicate customer rows:",
    customers_df.count() - customers_df.dropDuplicates().count()
)

print(
    "Duplicate order rows:",
    orders_df.count() - orders_df.dropDuplicates().count()
)

# Check duplicate primary IDs
print(
    "Duplicate customer IDs:",
    customers_df.count() - customers_df.select("customer_id").distinct().count()
)

print(
    "Duplicate order IDs:",
    orders_df.count() - orders_df.select("order_id").distinct().count()
)

Duplicate customer rows: 0
Duplicate order rows: 0
Duplicate customer IDs: 0
Duplicate order IDs: 0


In [0]:
# Find orders whose customer_id does not exist
# in the customers DataFrame

invalid_orders_df = orders_df.join(
    customers_df.select("customer_id"),
    on="customer_id",
    how="left_anti"
)

print("Orders with missing customers:", invalid_orders_df.count())

display(invalid_orders_df)

Orders with missing customers: 0


customer_id,order_id,product_name,product_category,quantity,unit_price,order_date,order_status


In [0]:
from pyspark.sql import functions as F

customers_clean = customers_df.select(
    F.col("customer_id"),
    F.trim(F.col("customer_name")).alias("customer_name"),
    F.trim(F.col("email")).alias("email"),
    F.trim(F.col("city")).alias("city"),
    F.trim(F.col("state")).alias("state"),
    F.to_date(F.col("signup_date")).alias("signup_date"),
    F.trim(F.col("customer_segment")).alias("customer_segment"),
    F.col("is_active")
)

orders_clean = orders_df.select(
    F.col("order_id"),
    F.col("customer_id"),
    F.trim(F.col("product_name")).alias("product_name"),
    F.trim(F.col("product_category")).alias("product_category"),
    F.col("quantity").cast("int").alias("quantity"),
    F.col("unit_price").cast("double").alias("unit_price"),
    F.to_date(F.col("order_date")).alias("order_date"),
    F.trim(F.col("order_status")).alias("order_status")
)

print("Cleaned customer rows:", customers_clean.count())
print("Cleaned order rows:", orders_clean.count())

Cleaned customer rows: 100
Cleaned order rows: 100


In [0]:
joined_df = customers_clean.join(
    orders_clean,
    on="customer_id",
    how="inner"
)

print("Joined rows:", joined_df.count())
print("Joined columns:", len(joined_df.columns))

display(joined_df)

Joined rows: 100
Joined columns: 15


customer_id,customer_name,email,city,state,signup_date,customer_segment,is_active,order_id,product_name,product_category,quantity,unit_price,order_date,order_status
1,Ravi Reddy,customer001@example.com,Pune,Maharashtra,2025-01-06,VIP,true,1072,Phone,Mobiles,2,1177.0,2026-08-02,Shipped
2,Sita Reddy,customer002@example.com,Vizag,Andhra Pradesh,2025-01-11,Premium,true,1015,Keyboard,Accessories,5,2868.0,2026-02-12,Pending
3,Kiran Reddy,customer003@example.com,Hyderabad,Telangana,2025-01-16,Regular,true,1058,Smartwatch,Wearables,3,8759.0,2026-06-21,Cancelled
4,Priya Reddy,customer004@example.com,Mumbai,Maharashtra,2025-01-21,New,true,1001,Laptop,Computers,1,950.0,2026-01-01,Delivered
5,Arun Reddy,customer005@example.com,Kolkata,West Bengal,2025-01-26,VIP,true,1044,Monitor,Computers,4,6841.0,2026-05-10,Shipped
6,Ananya Reddy,customer006@example.com,Jaipur,Rajasthan,2025-01-31,Premium,true,1087,Tablet,Mobiles,2,3232.0,2026-09-16,Pending
7,Rahul Reddy,customer007@example.com,Chennai,Tamil Nadu,2025-02-05,Regular,false,1030,Camera,Cameras,5,4923.0,2026-03-29,Cancelled
8,Sneha Reddy,customer008@example.com,Delhi,Delhi,2025-02-10,New,true,1073,Headphones,Audio,3,1314.0,2026-08-05,Delivered
9,Vikram Reddy,customer009@example.com,Coimbatore,Tamil Nadu,2025-02-15,VIP,true,1016,Mouse,Accessories,1,3005.0,2026-02-15,Shipped
10,Meera Reddy,customer010@example.com,Bangalore,Karnataka,2025-02-20,Premium,true,1059,Speaker,Audio,4,8896.0,2026-06-24,Pending


In [0]:
sales_df = (
    joined_df
    .withColumn(
        "total_amount",
        F.col("quantity") * F.col("unit_price")
    )
    .groupBy(
        "customer_id",
        "customer_name",
        "city"
    )
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.count("order_id").alias("total_orders")
    )
)

display(sales_df)

customer_id,customer_name,city,total_sales,total_orders
3,Kiran Reddy,Hyderabad,26277.0,1
6,Ananya Reddy,Jaipur,6464.0,1
45,Arun Kumar,Kolkata,38324.0,1
50,Meera Kumar,Bangalore,8544.0,1
59,Varun Kumar,Coimbatore,9855.0,1
91,Akhil Patel,Pune,13134.0,1
7,Rahul Reddy,Chennai,24615.0,1
39,Varun Sharma,Coimbatore,1635.0,1
51,Akhil Kumar,Pune,7654.0,1
60,Neha Kumar,Bangalore,8184.0,1


In [0]:
sales_df.orderBy(
    F.desc("total_sales")
).show()

+-----------+-------------+---------+-----------+------------+
|customer_id|customer_name|     city|total_sales|total_orders|
+-----------+-------------+---------+-----------+------------+
|         52|  Divya Kumar|    Vizag|    48590.0|           1|
|         17|  Manoj Reddy|  Chennai|    45165.0|           1|
|         82|   Sita Patel|    Vizag|    41740.0|           1|
|         45|   Arun Kumar|  Kolkata|    38324.0|           1|
|         47|  Rahul Kumar|  Chennai|    38315.0|           1|
|         10|  Meera Reddy|Bangalore|    35584.0|           1|
|         12|  Divya Reddy|    Vizag|    34890.0|           1|
|         75|   Sanjay Rao|  Kolkata|    32844.0|           1|
|         77|    Manoj Rao|  Chennai|    31465.0|           1|
|         40|  Neha Sharma|Bangalore|    30104.0|           1|
|         38|  Isha Sharma|    Delhi|    28332.0|           1|
|         42|   Sita Kumar|    Vizag|    28040.0|           1|
|          5|   Arun Reddy|  Kolkata|    27364.0|      

In [0]:
sales_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (23)
+- == Initial Plan ==
   PhotonResultStage (22)
   +- PhotonColumnarToRow (21)
      +- PhotonProject (20)
         +- PhotonGroupingAgg (19)
            +- PhotonShuffleExchangeSource (18)
               +- PhotonShuffleMapStage (17)
                  +- PhotonShuffleExchangeSink (16)
                     +- PhotonGroupingAgg (15)
                        +- PhotonProject (14)
                           +- PhotonBroadcastHashJoin Inner (13)
                              :- PhotonProject (4)
                              :  +- PhotonFilter (3)
                              :     +- PhotonRowToColumnar (2)
                              :        +- Scan csv  (1)
                              +- PhotonShuffleExchangeSource (12)
                                 +- PhotonShuffleMapStage (11)
                                    +- PhotonShuffleExchangeSink (10)
                                       +- PhotonGroupingAgg (9)
                          

In [0]:
sales_df.orderBy(F.desc("total_sales")).show()

+-----------+-------------+---------+-----------+------------+
|customer_id|customer_name|     city|total_sales|total_orders|
+-----------+-------------+---------+-----------+------------+
|         52|  Divya Kumar|    Vizag|    48590.0|           1|
|         17|  Manoj Reddy|  Chennai|    45165.0|           1|
|         82|   Sita Patel|    Vizag|    41740.0|           1|
|         45|   Arun Kumar|  Kolkata|    38324.0|           1|
|         47|  Rahul Kumar|  Chennai|    38315.0|           1|
|         10|  Meera Reddy|Bangalore|    35584.0|           1|
|         12|  Divya Reddy|    Vizag|    34890.0|           1|
|         75|   Sanjay Rao|  Kolkata|    32844.0|           1|
|         77|    Manoj Rao|  Chennai|    31465.0|           1|
|         40|  Neha Sharma|Bangalore|    30104.0|           1|
|         38|  Isha Sharma|    Delhi|    28332.0|           1|
|         42|   Sita Kumar|    Vizag|    28040.0|           1|
|          5|   Arun Reddy|  Kolkata|    27364.0|      

In [0]:
sales_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (23)
+- == Initial Plan ==
   PhotonResultStage (22)
   +- PhotonColumnarToRow (21)
      +- PhotonProject (20)
         +- PhotonGroupingAgg (19)
            +- PhotonShuffleExchangeSource (18)
               +- PhotonShuffleMapStage (17)
                  +- PhotonShuffleExchangeSink (16)
                     +- PhotonGroupingAgg (15)
                        +- PhotonProject (14)
                           +- PhotonBroadcastHashJoin Inner (13)
                              :- PhotonProject (4)
                              :  +- PhotonFilter (3)
                              :     +- PhotonRowToColumnar (2)
                              :        +- Scan csv  (1)
                              +- PhotonShuffleExchangeSource (12)
                                 +- PhotonShuffleMapStage (11)
                                    +- PhotonShuffleExchangeSink (10)
                                       +- PhotonGroupingAgg (9)
                          

In [0]:
filtered_df = sales_df.filter(
    sales_df.total_sales > 10000
)

filtered_df.show()

+-----------+-------------+---------+-----------+------------+
|customer_id|customer_name|     city|total_sales|total_orders|
+-----------+-------------+---------+-----------+------------+
|          3|  Kiran Reddy|Hyderabad|    26277.0|           1|
|         45|   Arun Kumar|  Kolkata|    38324.0|           1|
|         91|  Akhil Patel|     Pune|    13134.0|           1|
|          7|  Rahul Reddy|  Chennai|    24615.0|           1|
|         72|    Divya Rao|    Vizag|    21190.0|           1|
|         57|  Manoj Kumar|  Chennai|    11365.0|           1|
|         66|   Ananya Rao|   Jaipur|    19984.0|           1|
|         68|    Sneha Rao|    Delhi|    24222.0|           1|
|         97|  Manoj Patel|  Chennai|    25065.0|           1|
|          5|   Arun Reddy|  Kolkata|    27364.0|           1|
|         12|  Divya Reddy|    Vizag|    34890.0|           1|
|         27| Rahul Sharma|  Chennai|    18215.0|           1|
|          2|   Sita Reddy|    Vizag|    14340.0|      

In [0]:
test_df = sales_df.filter(
    sales_df.total_sales > 20000
)

In [0]:
test_df.show()

+-----------+-------------+---------+-----------+------------+
|customer_id|customer_name|     city|total_sales|total_orders|
+-----------+-------------+---------+-----------+------------+
|          3|  Kiran Reddy|Hyderabad|    26277.0|           1|
|         45|   Arun Kumar|  Kolkata|    38324.0|           1|
|          7|  Rahul Reddy|  Chennai|    24615.0|           1|
|         72|    Divya Rao|    Vizag|    21190.0|           1|
|         68|    Sneha Rao|    Delhi|    24222.0|           1|
|         97|  Manoj Patel|  Chennai|    25065.0|           1|
|          5|   Arun Reddy|  Kolkata|    27364.0|           1|
|         12|  Divya Reddy|    Vizag|    34890.0|           1|
|         10|  Meera Reddy|Bangalore|    35584.0|           1|
|         77|    Manoj Rao|  Chennai|    31465.0|           1|
|         75|   Sanjay Rao|  Kolkata|    32844.0|           1|
|         42|   Sita Kumar|    Vizag|    28040.0|           1|
|         17|  Manoj Reddy|  Chennai|    45165.0|      

In [0]:
sales_df.show()

+-----------+-------------+----------+-----------+------------+
|customer_id|customer_name|      city|total_sales|total_orders|
+-----------+-------------+----------+-----------+------------+
|          3|  Kiran Reddy| Hyderabad|    26277.0|           1|
|          6| Ananya Reddy|    Jaipur|     6464.0|           1|
|         45|   Arun Kumar|   Kolkata|    38324.0|           1|
|         50|  Meera Kumar| Bangalore|     8544.0|           1|
|         59|  Varun Kumar|Coimbatore|     9855.0|           1|
|         91|  Akhil Patel|      Pune|    13134.0|           1|
|          7|  Rahul Reddy|   Chennai|    24615.0|           1|
|         39| Varun Sharma|Coimbatore|     1635.0|           1|
|         51|  Akhil Kumar|      Pune|     7654.0|           1|
|         60|   Neha Kumar| Bangalore|     8184.0|           1|
|         72|    Divya Rao|     Vizag|    21190.0|           1|
|         89| Vikram Patel|Coimbatore|     8485.0|           1|
|          9| Vikram Reddy|Coimbatore|  

In [0]:
sales_df.count()

100

In [0]:
sales_df.printSchema()


root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- total_sales: double (nullable = true)
 |-- total_orders: long (nullable = false)



In [0]:
sales_df.select("customer_id").distinct().count()

100

In [0]:
sales_df.filter("customer_id IS NULL").count()

0

In [0]:
sales_df.filter("customer_name IS NULL").count()

0

In [0]:
sales_df.filter("city IS NULL").count()

0

In [0]:
sales_df.filter("total_sales IS NULL").count()

0

In [0]:
sales_df.count() - sales_df.dropDuplicates().count()

0

In [0]:
sales_df = sales_df.dropDuplicates()
sales_df.count()

100

In [0]:
sales_df = sales_df.withColumn(
    "total_sales",
    sales_df["total_sales"].cast("double")
)

sales_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- total_sales: double (nullable = true)
 |-- total_orders: long (nullable = false)



In [0]:
from pyspark.sql.functions import trim, col

sales_df = sales_df.withColumn(
    "city",
    trim(col("city"))
)

sales_df.show(5)

+-----------+-------------+----------+-----------+------------+
|customer_id|customer_name|      city|total_sales|total_orders|
+-----------+-------------+----------+-----------+------------+
|          3|  Kiran Reddy| Hyderabad|    26277.0|           1|
|          6| Ananya Reddy|    Jaipur|     6464.0|           1|
|         45|   Arun Kumar|   Kolkata|    38324.0|           1|
|         50|  Meera Kumar| Bangalore|     8544.0|           1|
|         59|  Varun Kumar|Coimbatore|     9855.0|           1|
+-----------+-------------+----------+-----------+------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import col, when

sales_df = sales_df.withColumn(
    "total_sales",
    when(col("total_sales") < 0, 0).otherwise(col("total_sales"))
)

sales_df.show(5)

+-----------+-------------+----------+-----------+------------+
|customer_id|customer_name|      city|total_sales|total_orders|
+-----------+-------------+----------+-----------+------------+
|          3|  Kiran Reddy| Hyderabad|    26277.0|           1|
|          6| Ananya Reddy|    Jaipur|     6464.0|           1|
|         45|   Arun Kumar|   Kolkata|    38324.0|           1|
|         50|  Meera Kumar| Bangalore|     8544.0|           1|
|         59|  Varun Kumar|Coimbatore|     9855.0|           1|
+-----------+-------------+----------+-----------+------------+
only showing top 5 rows


In [0]:
sales_df.groupBy("city").agg(
    {"total_sales": "sum"}
).orderBy("sum(total_sales)", ascending=False).show()

+----------+----------------+
|      city|sum(total_sales)|
+----------+----------------+
|     Vizag|        240650.0|
|   Chennai|        227400.0|
|   Kolkata|        187040.0|
| Bangalore|        176440.0|
|     Delhi|        136170.0|
| Hyderabad|        128220.0|
|    Jaipur|        101740.0|
|      Pune|         88040.0|
|Coimbatore|         49500.0|
|    Mumbai|         42650.0|
+----------+----------------+



In [0]:
from pyspark.sql.functions import col, sum, count

sales_df.groupBy("city").agg(
    sum("total_sales").alias("total_revenue"),
    count("*").alias("total_customers")
).orderBy(col("total_revenue").desc()).show()

+----------+-------------+---------------+
|      city|total_revenue|total_customers|
+----------+-------------+---------------+
|     Vizag|     240650.0|             10|
|   Chennai|     227400.0|             10|
|   Kolkata|     187040.0|             10|
| Bangalore|     176440.0|             10|
|     Delhi|     136170.0|             10|
| Hyderabad|     128220.0|             10|
|    Jaipur|     101740.0|             10|
|      Pune|      88040.0|             10|
|Coimbatore|      49500.0|             10|
|    Mumbai|      42650.0|             10|
+----------+-------------+---------------+



In [0]:
sales_df.write.format("delta").mode("overwrite").saveAsTable("sales_summary")

In [0]:
spark.table("sales_summary").show(5)

+-----------+-------------+----------+-----------+------------+
|customer_id|customer_name|      city|total_sales|total_orders|
+-----------+-------------+----------+-----------+------------+
|          3|  Kiran Reddy| Hyderabad|    26277.0|           1|
|          6| Ananya Reddy|    Jaipur|     6464.0|           1|
|         45|   Arun Kumar|   Kolkata|    38324.0|           1|
|         50|  Meera Kumar| Bangalore|     8544.0|           1|
|         59|  Varun Kumar|Coimbatore|     9855.0|           1|
+-----------+-------------+----------+-----------+------------+
only showing top 5 rows


In [0]:
spark.table("sales_summary").count()

100

In [0]:
spark.sql("DESCRIBE DETAIL sales_summary").show(truncate=False)

+------+------------------------------------+-------------------------------+-----------+--------+-----------------------+-------------------+----------------+-----------------+--------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                           |description|location|createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                                                                                                                                                                 |minReaderVersion|minWriterVersion|tableFeatures                            |statistics                   

In [0]:
from delta.tables import DeltaTable

DeltaTable.forName(
    spark,
    "sales_summary"
).history().show(truncate=False)

+-------+-------------------+--------------+-------------------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+-----------------+------------------------------------+------------------------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName                       |operation                        |operationParameters                                                                                                             

In [0]:
spark.sql("""
    SELECT *
    FROM sales_summary VERSION AS OF 0
""").show(5)

+-----------+-------------+----------+-----------+------------+
|customer_id|customer_name|      city|total_sales|total_orders|
+-----------+-------------+----------+-----------+------------+
|          3|  Kiran Reddy| Hyderabad|    26277.0|           1|
|          6| Ananya Reddy|    Jaipur|     6464.0|           1|
|         45|   Arun Kumar|   Kolkata|    38324.0|           1|
|         50|  Meera Kumar| Bangalore|     8544.0|           1|
|         59|  Varun Kumar|Coimbatore|     9855.0|           1|
+-----------+-------------+----------+-----------+------------+
only showing top 5 rows


In [0]:
spark.sql("""
    SELECT COUNT(*) AS total_rows
    FROM sales_summary VERSION AS OF 0
""").show()

+----------+
|total_rows|
+----------+
|       100|
+----------+

